In [1]:
#Import Libraries
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [2]:
#Load the dataset
df = pd.read_csv(r"C:\Users\palak\Downloads\dresses_dataset_50_rows.csv", encoding='latin-1')

In [3]:
df.head(5)

,Description,Silhouette,Fabric,Neckline,Sleeve,Length,Embellishment,Color,Category
0,Floor length chiffon bridesmaid dress with ple...,A-Line,Chiffon,V-Neck,Sleeveless,Floor,Pleated,Sage,Bridesmaid
1,Sparkly sequin fitted prom gown featuring a de...,Fitted,Sequin,Illusion,Sleeveless,Floor,Sequin,Silver,Prom
2,Off shoulder satin ball gown with corset bodic...,Ball Gown,Satin,Off Shoulder,Sleeveless,Floor,Corset,Royal Navy,Formal
3,Lace mermaid wedding dress with long sleeves a...,Mermaid,Lace,Round,Long,Floor,Lace,White,Wedding
4,Short cocktail dress with feather trim and bea...,Fit and Flare,Crepe,Round,Sleeveless,Short,Feather,Black,Cocktail


In [4]:
df.shape

(50, 9)

In [5]:
df.isnull().sum()

Description       0
Silhouette        0
Fabric            0
Neckline          0
Sleeve            0
Length            0
Embellishment    15
Color             0
Category          0
dtype: int64

In [6]:
df["Embellishment"] = df["Embellishment"].fillna("None")

In [7]:
df.isnull().sum()

Description      0
Silhouette       0
Fabric           0
Neckline         0
Sleeve           0
Length           0
Embellishment    0
Color            0
Category         0
dtype: int64

In [9]:
#create a function to clean the text data
def clean_text(text):

    text = str(text)   # Convert to string in case it's not

    text = text.lower()   # Convert to lowercase

    text = re.sub(r'[^a-zA-Z0-9 ]',' ',text)    # Remove special characters and punctuation

    text = re.sub(r'\s+',' ',text)    # Replace multiple spaces with a single space

    return text.strip()   # Remove leading and trailing spaces

In [10]:
df["Clean_Description"] = df["Description"].apply(clean_text)

In [11]:
df[["Description","Clean_Description"]].head()

,Description,Clean_Description
0,Floor length chiffon bridesmaid dress with ple...,floor length chiffon bridesmaid dress with ple...
1,Sparkly sequin fitted prom gown featuring a de...,sparkly sequin fitted prom gown featuring a de...
2,Off shoulder satin ball gown with corset bodic...,off shoulder satin ball gown with corset bodic...
3,Lace mermaid wedding dress with long sleeves a...,lace mermaid wedding dress with long sleeves a...
4,Short cocktail dress with feather trim and bea...,short cocktail dress with feather trim and bea...


In [12]:
# List of target columns
attributes = [
    "Silhouette",
    "Fabric",
    "Neckline",
    "Sleeve",
    "Length",
    "Embellishment",
    "Color",
    "Category"
]

# Create one TF-IDF vectorizer
vectorizer = TfidfVectorizer()

# Convert descriptions into features
X = vectorizer.fit_transform(df["Clean_Description"])

# Save vectorizer
joblib.dump(vectorizer, "vectorizer.pkl")

# Dictionary to store models
models = {}

# Train one model for each attribute
for attribute in attributes:
    print(attribute)
    print("Unique values:", df[attribute].nunique())
    print(df[attribute].value_counts())
    print("-" * 50)

    # Target column
    y = df[attribute]

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    # Create model
    model = LogisticRegression(max_iter=1000)

    # Train model
    model.fit(X_train, y_train)

    # Prediction
    y_pred = model.predict(X_test)

    # Evaluation
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")

    print("Accuracy :", round(accuracy, 2))
    print("F1 Score :", round(f1, 2))
    print("\nClassification Report")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("\nConfusion Matrix")
    print(confusion_matrix(y_test, y_pred))

    # Save model
    joblib.dump(model, f"{attribute.lower()}_model.pkl")

    # Store model in dictionary
    models[attribute] = model

print("\nAll models trained and saved successfully!")

Silhouette
Unique values: 9
Silhouette
A-Line           15
Sheath           10
Ball Gown         8
Mermaid           5
Fitted            4
Fit and Flare     3
Wrap              2
Bodycon           2
Shirt             1
Name: count, dtype: int64
--------------------------------------------------
Accuracy : 0.1
F1 Score : 0.04

Classification Report
               precision    recall  f1-score   support

       A-Line       0.11      0.50      0.18         2
    Ball Gown       0.00      0.00      0.00         4
Fit and Flare       0.00      0.00      0.00         1
      Mermaid       0.00      0.00      0.00         2
       Sheath       0.00      0.00      0.00         1

     accuracy                           0.10        10
    macro avg       0.02      0.10      0.04        10
 weighted avg       0.02      0.10      0.04        10


Confusion Matrix
[[1 0 0 0 1]
 [4 0 0 0 0]
 [1 0 0 0 0]
 [2 0 0 0 0]
 [1 0 0 0 0]]
Fabric
Unique values: 13
Fabric
Chiffon    6
Satin      6
Lace      

C:\Users\palak\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:1469: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  check_classification_targets(y)


In [13]:
#Test on new data
description = "Elegant satin ball gown with off shoulder neckline in royal navy."

description = clean_text(description)

X_new = vectorizer.transform([description])

for attribute in attributes:
    model = joblib.load(f"{attribute.lower()}_model.pkl")
    prediction = model.predict(X_new)[0]
    print(f"{attribute}: {prediction}")

Silhouette: Ball Gown
Fabric: Satin
Neckline: Off Shoulder
Sleeve: Sleeveless
Length: Floor
Embellishment: None
Color: White
Category: Wedding
